<a href="https://colab.research.google.com/github/HoangVo-Prog/Basic-Data-Crawling/blob/main/pytube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q pytubefix ffmpeg
from pytubefix import YouTube
import os
import shutil
from tqdm import tqdm

In [6]:
def get_video_resolution(path):
    # Use ffprobe to extract resolution (width,height)
    try:
        import subprocess
        result = subprocess.run(
            [
                'ffprobe', '-v', 'error',
                '-select_streams', 'v:0',
                '-show_entries', 'stream=width,height',
                '-of', 'csv=p=0', path
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        width, height = map(int, result.stdout.strip().split(','))
        return width, height
    except Exception as e:
        print(f"Could not read resolution: {e}")
        return None, None

def download_and_merge_video_audio(url, resolution='720p', drive_path='/content/drive/MyDrive/'):
    try:
        yt = YouTube(url)

        # Get video-only and audio-only streams
        video_stream = yt.streams.filter(res=resolution, mime_type='video/mp4', progressive=False).first()
        audio_stream = yt.streams.filter(only_audio=True, mime_type='audio/mp4').order_by('abr').desc().first()

        if video_stream and audio_stream:
            video_path = video_stream.download(filename='video.mp4')
            audio_path = audio_stream.download(filename='audio.mp4')

            merged_path = "/content/merged.mp4"
            os.system(f'ffmpeg -y -i "{video_path}" -i "{audio_path}" -c copy "{merged_path}"')

            # 🔍 Check resolution before moving
            width, height = get_video_resolution(merged_path)
            print(f"Actual resolution: {width}x{height}")

            if height and height >= 700:
                video_id = video_url.split('=')[-1]

                final_path = os.path.join(drive_path, f"{video_id}.mp4")
                shutil.move(merged_path, final_path)
                print(f"✅ Video successfully saved to Google Drive at: {final_path}")
            else:
                print("⚠️ Skipped: Video resolution is below 720p threshold.")
                os.remove(merged_path)

            os.remove(video_path)
            os.remove(audio_path)

        else:
            print("720p video-only stream or audio stream not available.")
    except Exception as e:
        print(f"Error: {e}")


Downloading: Nebula - Cyber Rush【Nightcore Music】
Actual resolution: 1280x718
✅ Video successfully saved to Google Drive at: /content/drive/MyDrive/92w4Ru80VXc_720p.mp4


In [ ]:
video_url = 'https://www.youtube.com/watch?v=92w4Ru80VXc'
download_and_merge_video_audio(video_url)

In [ ]:
# video_urls = '/content/drive/MyDrive/video_urls'
# drive_path = '/content/drive/MyDrive/videos'
# for video_url in tqdm(os.listdir(video_urls)):
#   download_and_merge_video_audio(video_url, '720p', drive_path)